# NEXUS — Fase 4: Fine-tuning RoBERTa
## TCC: Detecção de Fake News — Generalização, Robustez e NLP Moderno

### Papel de cada dataset neste notebook

| Dataset | Tipo de texto | Avg words | Papel |
|---|---|---|---|
| kaggle_fakenews | Artigo completo | 411 | **Treino primário** + in-domain eval |
| gossipcop | Artigo completo | 604 | **Treino secundário** + in-domain eval |
| liar | Claim curto | 18 | **Apenas teste** cross-domain |
| politifact | Claim curto | 16 | **Apenas teste** cross-domain |

**Por que LIAR e PolitiFact não são usados para treino?**
Com média de 16-18 palavras, fine-tunar RoBERTa nesses datasets
desperdiça 494 dos 512 tokens disponíveis. O modelo aprende quase nada
de semântica profunda. Eles existem para medir generalização cross-domain.

### Experimentos executados (mesmos da Fase 3B para comparação direta)
- **[A] In-domain**: kaggle_fakenews, gossipcop
- **[B] Cross-domain**: kaggle→liar, kaggle→gossipcop, kaggle→politifact, gossipcop→liar
- **[C] LODO**: test=gossipcop, test=liar, test=politifact

### Tempo estimado por experimento (T4 GPU)
- kaggle_fakenews in-domain: ~90 min
- gossipcop in-domain: ~60 min
- Cross-domain (kaggle como treino): ~90 min cada
- LODO: ~120 min cada

In [ ]:
# ── Instalações ──────────────────────────────────────────────────────────
!pip install transformers==4.44.2 accelerate==0.33.0 -q

import os, sys, json, time, warnings, logging
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import RobertaTokenizer, RobertaModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    f1_score, matthews_corrcoef, roc_auc_score,
    average_precision_score, brier_score_loss,
    confusion_matrix, classification_report
)
from sklearn.linear_model import LogisticRegression

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(message)s')
logger = logging.getLogger(__name__)

# ── Reprodutibilidade ────────────────────────────────────────────────────
SEED = 42
import random
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
os.environ['PYTHONHASHSEED'] = str(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('AVISO: Sem GPU. Ative em Settings > Accelerator > GPU T4 x2')

## Configuração — EDITE AQUI antes de rodar

In [ ]:
# ════════════════════════════════════════════════════════════════════════
# CONFIGURAÇÃO DO EXPERIMENTO
# Execute uma célula de configuração por sessão Kaggle
# ════════════════════════════════════════════════════════════════════════

# Escolha UM dos blocos abaixo e comente os outros

# ── BLOCO A1: In-domain Kaggle (Sessão 1 — rodar primeiro) ──────────────
EXPERIMENT_TYPE  = 'indomain'
TRAIN_DATASETS   = ['kaggle_fakenews']   # datasets usados para treino
TEST_DATASET     = 'kaggle_fakenews'     # dataset de teste
EXP_ID           = 'roberta_indomain_kaggle_fakenews'

# ── BLOCO A2: In-domain GossipCop ───────────────────────────────────────
# EXPERIMENT_TYPE  = 'indomain'
# TRAIN_DATASETS   = ['gossipcop']
# TEST_DATASET     = 'gossipcop'
# EXP_ID           = 'roberta_indomain_gossipcop'

# ── BLOCO B1: Cross-domain Kaggle → LIAR (mais importante) ─────────────
# EXPERIMENT_TYPE  = 'crossdomain'
# TRAIN_DATASETS   = ['kaggle_fakenews']
# TEST_DATASET     = 'liar'
# EXP_ID           = 'roberta_kaggle_to_liar'

# ── BLOCO B2: Cross-domain Kaggle → GossipCop ──────────────────────────
# EXPERIMENT_TYPE  = 'crossdomain'
# TRAIN_DATASETS   = ['kaggle_fakenews']
# TEST_DATASET     = 'gossipcop'
# EXP_ID           = 'roberta_kaggle_to_gossipcop'

# ── BLOCO B3: Cross-domain Kaggle → PolitiFact ─────────────────────────
# EXPERIMENT_TYPE  = 'crossdomain'
# TRAIN_DATASETS   = ['kaggle_fakenews']
# TEST_DATASET     = 'politifact'
# EXP_ID           = 'roberta_kaggle_to_politifact'

# ── BLOCO B4: Cross-domain GossipCop → LIAR ────────────────────────────
# EXPERIMENT_TYPE  = 'crossdomain'
# TRAIN_DATASETS   = ['gossipcop']
# TEST_DATASET     = 'liar'
# EXP_ID           = 'roberta_gossipcop_to_liar'

# ── BLOCO C1: LODO — test=GossipCop (CRÍTICO: MCC=-0.02 no baseline) ───
# EXPERIMENT_TYPE  = 'lodo'
# TRAIN_DATASETS   = ['kaggle_fakenews', 'liar', 'politifact']  # todos exceto gossipcop
# TEST_DATASET     = 'gossipcop'
# EXP_ID           = 'roberta_lodo_test_gossipcop'

# ── BLOCO C2: LODO — test=LIAR ─────────────────────────────────────────
# EXPERIMENT_TYPE  = 'lodo'
# TRAIN_DATASETS   = ['kaggle_fakenews', 'gossipcop', 'politifact']
# TEST_DATASET     = 'liar'
# EXP_ID           = 'roberta_lodo_test_liar'

# ── BLOCO C3: LODO — test=PolitiFact ───────────────────────────────────
# EXPERIMENT_TYPE  = 'lodo'
# TRAIN_DATASETS   = ['kaggle_fakenews', 'gossipcop', 'liar']
# TEST_DATASET     = 'politifact'
# EXP_ID           = 'roberta_lodo_test_politifact'

# ════════════════════════════════════════════════════════════════════════
# HIPERPARÂMETROS — não alterar entre experimentos (comparabilidade)
# ════════════════════════════════════════════════════════════════════════
MODEL_NAME    = 'roberta-base'
MAX_LENGTH    = 512
HEAD_LEN      = 128    # tokens do início para estratégia head+tail
BATCH_SIZE    = 16     # reduzir para 8 se OOM
LEARNING_RATE = 2e-5
NUM_EPOCHS    = 4
WARMUP_RATIO  = 0.10
WEIGHT_DECAY  = 0.01
DROPOUT       = 0.1
FP16          = True
PATIENCE      = 2      # early stopping por Macro F1

# Diretórios
DATA_DIR   = Path('/kaggle/input/nexus-processed')
OUTPUT_DIR = Path('/kaggle/working/nexus_roberta')
CKPT_DIR   = OUTPUT_DIR / 'checkpoints'
FIG_DIR    = OUTPUT_DIR / 'figures'
for d in [OUTPUT_DIR, CKPT_DIR, FIG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f'Experimento configurado: {EXP_ID}')
print(f'Tipo: {EXPERIMENT_TYPE}')
print(f'Treino em: {TRAIN_DATASETS}')
print(f'Teste em:  {TEST_DATASET}')

## Carregamento e Splits

In [ ]:
def load_parquet(name: str) -> pd.DataFrame:
    """
    Carrega um parquet processado.
    Detecta automaticamente o path correto dentro do Kaggle.
    """
    candidates = [
        DATA_DIR / f'{name}.parquet',
        Path(f'/kaggle/input/{name}/{name}.parquet'),
    ]
    for path in candidates:
        if path.exists():
            df = pd.read_parquet(path)
            break
    else:
        # Listar o que existe para debug
        print('Arquivos disponíveis em /kaggle/input:')
        for root, dirs, files in os.walk('/kaggle/input'):
            for f in files:
                if f.endswith('.parquet'):
                    print(f'  {os.path.join(root, f)}')
        raise FileNotFoundError(f"'{name}.parquet' não encontrado. Ajuste DATA_DIR.")

    # Selecionar coluna de texto correta
    for col in ['text_cleaned', 'input_text', 'text']:
        if col in df.columns:
            df['input_text'] = df[col].fillna('').astype(str)
            break

    # Limpeza básica
    mask = (df['input_text'].str.split().str.len() >= 3) & df['label'].notna()
    df   = df[mask].reset_index(drop=True)
    df['label'] = df['label'].astype(int)

    n_fake = (df['label'] == 0).sum()
    n_real = (df['label'] == 1).sum()
    avg_w  = df['input_text'].str.split().str.len().mean()
    print(f'[{name}] n={len(df):,} | fake={n_fake:,} ({n_fake/len(df):.1%}) '
          f'| real={n_real:,} | avg_words={avg_w:.0f}')
    return df


def stratified_split(df, train_r=0.70, val_r=0.10, test_r=0.20, seed=SEED):
    """Split estratificado por classe. Garante proporção igual em todos os splits."""
    rng = np.random.default_rng(seed)
    train_parts, val_parts, test_parts = [], [], []
    for lbl in [0, 1]:
        idx = df[df['label'] == lbl].index.tolist()
        rng.shuffle(idx)
        n   = len(idx)
        ntr = int(n * train_r)
        nv  = int(n * val_r)
        train_parts.append(df.loc[idx[:ntr]])
        val_parts.append(df.loc[idx[ntr:ntr+nv]])
        test_parts.append(df.loc[idx[ntr+nv:]])
    train = pd.concat(train_parts).sample(frac=1, random_state=seed).reset_index(drop=True)
    val   = pd.concat(val_parts  ).sample(frac=1, random_state=seed).reset_index(drop=True)
    test  = pd.concat(test_parts ).sample(frac=1, random_state=seed).reset_index(drop=True)
    print(f'  Split: train={len(train):,} | val={len(val):,} | test={len(test):,}')
    return train, val, test


def cross_domain_split(all_dfs, train_names, test_name, val_r=0.10, seed=SEED):
    """
    Treina em train_names, testa em test_name.
    Val vem de uma fração estratificada do treino (mesmo domínio).
    CRÍTICO: vocabulário do teste nunca contamina o treino.
    """
    rng       = np.random.default_rng(seed)
    all_train = pd.concat([all_dfs[n] for n in train_names], ignore_index=True)
    test      = all_dfs[test_name].copy()

    val_parts, tr_parts = [], []
    for lbl in [0, 1]:
        idx = all_train[all_train['label'] == lbl].index.tolist()
        rng.shuffle(idx)
        nv  = max(1, int(len(idx) * val_r))
        val_parts.append(all_train.loc[idx[:nv]])
        tr_parts.append(all_train.loc[idx[nv:]])

    train = pd.concat(tr_parts ).sample(frac=1, random_state=seed).reset_index(drop=True)
    val   = pd.concat(val_parts).sample(frac=1, random_state=seed).reset_index(drop=True)

    print(f'  Train sources: {train_names} | Test source: {test_name}')
    print(f'  train={len(train):,} | val={len(val):,} | test={len(test):,}')
    return train, val, test


# ── Carregar todos os datasets necessários ───────────────────────────────
print('Carregando datasets...')
all_datasets = {}
datasets_needed = list(set(TRAIN_DATASETS + [TEST_DATASET]))
for name in datasets_needed:
    all_datasets[name] = load_parquet(name)

# ── Montar splits conforme tipo de experimento ───────────────────────────
print(f'\nMontando splits para: {EXPERIMENT_TYPE}')
if EXPERIMENT_TYPE == 'indomain':
    train, val, test = stratified_split(all_datasets[TEST_DATASET])

elif EXPERIMENT_TYPE in ('crossdomain', 'lodo'):
    train, val, test = cross_domain_split(all_datasets, TRAIN_DATASETS, TEST_DATASET)

print(f'\nDataset de TESTE: {TEST_DATASET}')
print(f'  Distribuição test: fake={(test["label"]==0).sum():,} | real={(test["label"]==1).sum():,}')

## Dataset PyTorch — Estratégia Head+Tail

In [ ]:
class FakeNewsDataset(Dataset):
    """
    Estratégia head+tail para textos longos:
      - Primeiros HEAD_LEN tokens (lede jornalístico)
      - Últimos (512-2-HEAD_LEN) tokens (conclusão/desfecho)

    Por que isso importa para fake news?
    Notícias falsas frequentemente têm padrões tanto na abertura
    (afirmações sensacionalistas) quanto no fechamento
    (call-to-action, apelos emocionais). Truncar só o início perde
    esses sinais — especialmente relevante para GossipCop (avg 604 palavras).

    Para LIAR/PolitiFact (avg 16-18 palavras): texto cabe inteiro,
    estratégia não se aplica — mas o código lida corretamente.
    """
    def __init__(self, texts, labels, tokenizer, max_len=MAX_LENGTH, head_len=HEAD_LEN):
        self.texts    = [str(t) for t in texts]
        self.labels   = list(labels)
        self.tok      = tokenizer
        self.max_len  = max_len
        self.head_len = head_len
        self.tail_len = max_len - 2 - head_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            add_special_tokens=False, truncation=False,
            return_attention_mask=False,
        )
        ids = enc['input_ids']

        # Truncamento head+tail
        if len(ids) > self.max_len - 2:
            ids = ids[:self.head_len] + ids[-self.tail_len:]

        # [CLS] + ids + [SEP] + padding
        ids  = [self.tok.cls_token_id] + ids + [self.tok.sep_token_id]
        pad  = self.max_len - len(ids)
        attn = [1] * len(ids) + [0] * pad
        ids  = ids + [self.tok.pad_token_id] * pad

        return {
            'input_ids':      torch.tensor(ids,  dtype=torch.long),
            'attention_mask': torch.tensor(attn, dtype=torch.long),
            'labels':         torch.tensor(self.labels[idx], dtype=torch.long),
        }


print('Carregando tokenizer RoBERTa...')
tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

# Análise de truncamento — quanto texto é perdido?
sample_lengths = train['input_text'].sample(min(500, len(train)), random_state=42)\
                      .apply(lambda t: len(tokenizer.encode(t, add_special_tokens=False)))
print(f'\nAnálise de tokens no conjunto de treino (amostra {len(sample_lengths)}x):')
print(f'  Mediana: {sample_lengths.median():.0f} tokens')
print(f'  P90:     {sample_lengths.quantile(0.9):.0f} tokens')
print(f'  % acima de 512: {(sample_lengths > 512).mean():.1%} (serão truncados head+tail)')


def build_loaders(train_df, val_df, test_df):
    """DataLoaders com weighted sampler no treino."""
    # Weighted sampler: compensa desbalanceamento por classe
    # Essencial para LODO com PolitiFact (83% fake) e GossipCop (24% fake)
    labels       = train_df['label'].values
    class_counts = np.bincount(labels)
    weights      = torch.tensor(1.0 / class_counts[labels], dtype=torch.float)
    sampler      = WeightedRandomSampler(weights, num_samples=len(labels), replacement=True)

    ds_tr  = FakeNewsDataset(train_df['input_text'], train_df['label'], tokenizer)
    ds_val = FakeNewsDataset(val_df['input_text'],   val_df['label'],   tokenizer)
    ds_te  = FakeNewsDataset(test_df['input_text'],  test_df['label'],  tokenizer)

    pin = DEVICE == 'cuda'
    ld_tr  = DataLoader(ds_tr,  batch_size=BATCH_SIZE,   sampler=sampler,  num_workers=2, pin_memory=pin)
    ld_val = DataLoader(ds_val, batch_size=BATCH_SIZE*2, shuffle=False,    num_workers=2, pin_memory=pin)
    ld_te  = DataLoader(ds_te,  batch_size=BATCH_SIZE*2, shuffle=False,    num_workers=2, pin_memory=pin)

    print(f'DataLoaders: train={len(ld_tr)} batches | val={len(ld_val)} | test={len(ld_te)}')
    return ld_tr, ld_val, ld_te


ld_train, ld_val, ld_test = build_loaders(train, val, test)
print('Datasets e DataLoaders prontos.')

## Modelo RoBERTa

In [ ]:
class RoBERTaClassifier(nn.Module):
    """
    RoBERTa para classificação binária.

    Arquitetura: encoder → [CLS] → Dropout → Linear(768, 2)

    Sem camadas intermediárias densas: Sun et al. (2019) demonstram
    que para fine-tuning de BERT/RoBERTa em classificação, adicionar
    camadas intermediárias aumenta overfitting sem ganho de F1.
    """
    def __init__(self):
        super().__init__()
        self.roberta    = RobertaModel.from_pretrained(MODEL_NAME)
        self.dropout    = nn.Dropout(DROPOUT)
        self.classifier = nn.Linear(self.roberta.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask, labels=None):
        out    = self.roberta(input_ids=input_ids, attention_mask=attention_mask)
        pooled = out.last_hidden_state[:, 0, :]  # token [CLS]
        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)
        result = {'logits': logits}
        if labels is not None:
            result['loss'] = nn.CrossEntropyLoss()(logits, labels)
        return result


print('Inicializando modelo...')
model = RoBERTaClassifier().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Parâmetros treináveis: {n_params:,}')
print(f'Uso de memória estimado: ~{n_params * 4 / 1e9:.1f} GB (fp32) / ~{n_params * 2 / 1e9:.1f} GB (fp16)')

## Training Loop

In [ ]:
def run_training(model, ld_train, ld_val, exp_id):
    """
    Training loop com:
    - AdamW com weight decay separado para bias/LayerNorm
    - Linear warmup (10% steps) + linear decay
    - Gradient clipping (max_norm=1.0)
    - Mixed precision fp16
    - Early stopping por Macro F1 (não loss)
    - Checkpoint do melhor modelo por Macro F1

    Por que checkpoint por Macro F1 e não loss?
    Em datasets desbalanceados, a loss pode cair enquanto o modelo
    colapsa para a classe majoritária. Macro F1 detecta esse problema.
    """
    no_decay  = ['bias', 'LayerNorm.weight']
    optimizer = torch.optim.AdamW([
        {'params': [p for n,p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         'weight_decay': WEIGHT_DECAY},
        {'params': [p for n,p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         'weight_decay': 0.0},
    ], lr=LEARNING_RATE)

    total_steps  = len(ld_train) * NUM_EPOCHS
    warmup_steps = int(total_steps * WARMUP_RATIO)
    scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    scaler       = torch.cuda.amp.GradScaler() if FP16 and DEVICE == 'cuda' else None

    history     = {'train_loss': [], 'val_loss': [], 'val_macro_f1': [], 'val_mcc': []}
    best_f1     = 0.0
    n_bad       = 0
    ckpt_path   = CKPT_DIR / f'{exp_id}_best.pt'
    t_start     = time.time()

    for epoch in range(NUM_EPOCHS):
        # ── Treino ──────────────────────────────────────────────────────
        model.train()
        tr_loss = 0.0
        for step, batch in enumerate(ld_train):
            ids  = batch['input_ids'].to(DEVICE)
            attn = batch['attention_mask'].to(DEVICE)
            lbls = batch['labels'].to(DEVICE)
            optimizer.zero_grad()
            if scaler:
                with torch.cuda.amp.autocast():
                    out = model(ids, attn, lbls)
                scaler.scale(out['loss']).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
            else:
                out = model(ids, attn, lbls)
                out['loss'].backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            scheduler.step()
            tr_loss += out['loss'].item()
            if (step + 1) % 200 == 0:
                elapsed = (time.time() - t_start) / 60
                print(f'  E{epoch+1} step {step+1}/{len(ld_train)} '
                      f'loss={out["loss"].item():.4f} '
                      f'lr={scheduler.get_last_lr()[0]:.2e} '
                      f'({elapsed:.1f}min)')
        tr_loss /= len(ld_train)

        # ── Validação ────────────────────────────────────────────────────
        model.eval()
        vl, preds, probs, lbls_all = 0.0, [], [], []
        with torch.no_grad():
            for batch in ld_val:
                ids  = batch['input_ids'].to(DEVICE)
                attn = batch['attention_mask'].to(DEVICE)
                l    = batch['labels'].to(DEVICE)
                ctx  = torch.cuda.amp.autocast() if scaler else torch.no_grad()
                with (torch.cuda.amp.autocast() if scaler else torch.no_grad()):
                    out  = model(ids, attn, l)
                vl      += out['loss'].item()
                preds.extend(out['logits'].argmax(-1).cpu().numpy())
                probs.extend(torch.softmax(out['logits'], -1)[:,1].cpu().numpy())
                lbls_all.extend(l.cpu().numpy())
        vl    /= len(ld_val)
        val_f1 = f1_score(lbls_all, preds, average='macro', zero_division=0)
        val_mc = matthews_corrcoef(lbls_all, preds)

        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl)
        history['val_macro_f1'].append(val_f1)
        history['val_mcc'].append(val_mc)

        elapsed = (time.time() - t_start) / 60
        print(f'Época {epoch+1}/{NUM_EPOCHS} | '
              f'tr_loss={tr_loss:.4f} | vl_loss={vl:.4f} | '
              f'val_F1={val_f1:.4f} | val_MCC={val_mc:.4f} | '
              f'{elapsed:.1f}min')

        if val_f1 > best_f1:
            best_f1 = val_f1
            n_bad   = 0
            torch.save(model.state_dict(), ckpt_path)
            print(f'  ✓ Novo melhor modelo (F1={val_f1:.4f}) salvo')
        else:
            n_bad += 1
            if n_bad >= PATIENCE:
                print(f'  Early stopping na época {epoch+1}')
                break

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    history['best_val_f1'] = best_f1
    print(f'\nTreino concluído. Melhor val F1={best_f1:.4f} | Tempo={(time.time()-t_start)/60:.1f}min')
    return history, val_f1


@torch.no_grad()
def evaluate(model, loader):
    """Avalia e retorna (preds, probs_classe1, labels_verdadeiros)."""
    model.eval()
    preds, probs, lbls = [], [], []
    for batch in loader:
        ids  = batch['input_ids'].to(DEVICE)
        attn = batch['attention_mask'].to(DEVICE)
        with (torch.cuda.amp.autocast() if FP16 and DEVICE == 'cuda' else torch.no_grad()):
            out = model(ids, attn)
        preds.extend(out['logits'].argmax(-1).cpu().numpy())
        probs.extend(torch.softmax(out['logits'], -1)[:,1].cpu().numpy())
        lbls.extend(batch['labels'].numpy())
    return np.array(preds), np.array(probs), np.array(lbls)


print('Training loop definido.')

## Executar Treinamento

In [ ]:
print('=' * 60)
print(f'INICIANDO: {EXP_ID}')
print('=' * 60)
history, best_val_f1 = run_training(model, ld_train, ld_val, EXP_ID)

## Avaliação Completa + Calibração

In [ ]:
def compute_ece(y_true, y_prob, n_bins=15):
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    bd   = {'centers': [], 'acc': [], 'conf': [], 'counts': []}
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (y_prob >= lo) & (y_prob < hi)
        if m.sum() == 0:
            continue
        acc  = float(y_true[m].mean())
        conf = float(y_prob[m].mean())
        cnt  = int(m.sum())
        ece += (cnt / len(y_true)) * abs(acc - conf)
        bd['centers'].append((lo + hi) / 2)
        bd['acc'].append(acc)
        bd['conf'].append(conf)
        bd['counts'].append(cnt)
    return float(ece), bd


# ── Avaliação no val (para Platt Scaling) ────────────────────────────────
_, val_probs, val_labels = evaluate(model, ld_val)

# ── Avaliação no test ─────────────────────────────────────────────────────
test_preds, test_probs_raw, test_labels = evaluate(model, ld_test)

# ── Platt Scaling (ajustado no val, aplicado no test) ────────────────────
platt = LogisticRegression(C=1e10, solver='lbfgs', max_iter=1000)
platt.fit(val_probs.reshape(-1, 1), val_labels)
test_probs_cal = platt.predict_proba(test_probs_raw.reshape(-1, 1))[:, 1]

# ── Métricas ──────────────────────────────────────────────────────────────
ece_raw, _       = compute_ece(test_labels, test_probs_raw)
ece_cal, bd_cal  = compute_ece(test_labels, test_probs_cal)

metrics = {
    'experiment_id':   EXP_ID,
    'experiment_type': EXPERIMENT_TYPE,
    'train_datasets':  TRAIN_DATASETS,
    'test_dataset':    TEST_DATASET,
    'model':           MODEL_NAME,
    'n_train': len(train), 'n_val': len(val), 'n_test': len(test),
    'n_fake_test': int((test_labels == 0).sum()),
    'n_real_test': int((test_labels == 1).sum()),
    'macro_f1':  float(f1_score(test_labels, test_preds, average='macro',    zero_division=0)),
    'f1_fake':   float(f1_score(test_labels, test_preds, pos_label=0,        zero_division=0)),
    'f1_real':   float(f1_score(test_labels, test_preds, pos_label=1,        zero_division=0)),
    'mcc':       float(matthews_corrcoef(test_labels, test_preds)),
    'roc_auc':   float(roc_auc_score(test_labels, test_probs_cal)),
    'pr_auc':    float(average_precision_score(test_labels, test_probs_cal)),
    'brier':     float(brier_score_loss(test_labels, test_probs_cal)),
    'ece_before_calibration': ece_raw,
    'ece_after_calibration':  ece_cal,
    'hyperparams': {
        'max_length': MAX_LENGTH, 'head_len': HEAD_LEN,
        'batch_size': BATCH_SIZE, 'lr': LEARNING_RATE,
        'epochs': NUM_EPOCHS,    'dropout': DROPOUT,
    },
    'history': history,
    'bin_data_calibrated': bd_cal,
}

print('\n' + '=' * 60)
print(f'RESULTADOS — {EXP_ID}')
print('=' * 60)
print(f"  Macro F1  : {metrics['macro_f1']:.4f}")
print(f"  MCC       : {metrics['mcc']:.4f}")
print(f"  ROC-AUC   : {metrics['roc_auc']:.4f}")
print(f"  PR-AUC    : {metrics['pr_auc']:.4f}")
print(f"  ECE antes : {ece_raw:.4f}")
print(f"  ECE depois: {ece_cal:.4f}  (Platt Scaling)")
print(f"  Brier     : {metrics['brier']:.4f}")
print()
print(classification_report(
    test_labels, test_preds,
    target_names=['Fake (0)', 'Real (1)']
))

# Comparação direta com baseline TF-IDF+LR da Fase 3B
BASELINE_REF = {
    'roberta_indomain_kaggle_fakenews': {'macro_f1': 0.9883, 'mcc': 0.9766},
    'roberta_indomain_gossipcop':       {'macro_f1': 0.7980, 'mcc': 0.5960},
    'roberta_kaggle_to_liar':           {'macro_f1': 0.3658, 'mcc': 0.0369},
    'roberta_kaggle_to_gossipcop':      {'macro_f1': 0.2024, 'mcc': 0.0119},
    'roberta_kaggle_to_politifact':     {'macro_f1': 0.4947, 'mcc': 0.0209},
    'roberta_gossipcop_to_liar':        {'macro_f1': 0.4100, 'mcc': 0.0225},
    'roberta_lodo_test_gossipcop':      {'macro_f1': 0.2089, 'mcc': -0.0205},
    'roberta_lodo_test_liar':           {'macro_f1': 0.4898, 'mcc': 0.0744},
    'roberta_lodo_test_politifact':     {'macro_f1': 0.4578, 'mcc': 0.0083},
}

if EXP_ID in BASELINE_REF:
    ref = BASELINE_REF[EXP_ID]
    delta_f1  = metrics['macro_f1'] - ref['macro_f1']
    delta_mcc = metrics['mcc']      - ref['mcc']
    print(f'Comparação com TF-IDF+LR (Fase 3B):')
    print(f'  Δ Macro F1 = {delta_f1:+.4f}  ({"RoBERTa melhor" if delta_f1 > 0 else "TF-IDF melhor"})')
    print(f'  Δ MCC      = {delta_mcc:+.4f}')

## Visualizações

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Curvas de aprendizado ─────────────────────────────────────────────────
ax  = axes[0]
ep  = range(1, len(history['train_loss']) + 1)
ax.plot(ep, history['train_loss'], 'b-o', label='Train Loss',  linewidth=2)
ax.plot(ep, history['val_loss'],   'r-o', label='Val Loss',    linewidth=2)
ax.set_xlabel('Época'); ax.set_ylabel('Loss')
ax.set_title('Curvas de Aprendizado', fontweight='bold')
ax.legend(loc='upper right')
ax2 = ax.twinx()
ax2.plot(ep, history['val_macro_f1'], 'g--s', label='Val Macro F1', linewidth=2)
ax2.set_ylabel('Macro F1', color='green')
ax2.legend(loc='lower right')

# ── Reliability Diagram ───────────────────────────────────────────────────
ax = axes[1]
ax.plot([0,1], [0,1], 'k--', alpha=0.5, label='Calibração perfeita', lw=1.5)
if bd_cal['centers']:
    gaps = [abs(a-c) for a,c in zip(bd_cal['acc'], bd_cal['conf'])]
    bottoms = [min(a,c) for a,c in zip(bd_cal['acc'], bd_cal['conf'])]
    colors  = ['#E74C3C' if a < c else '#2ECC71'
                for a,c in zip(bd_cal['acc'], bd_cal['conf'])]
    for ctr, gap, bot, col in zip(bd_cal['centers'], gaps, bottoms, colors):
        ax.bar(ctr, gap, bottom=bot, width=0.05, alpha=0.35, color=col)
    sc = ax.scatter(bd_cal['conf'], bd_cal['acc'],
                    c=bd_cal['counts'], cmap='Blues', s=80, zorder=5,
                    edgecolors='gray', linewidths=0.5)
    plt.colorbar(sc, ax=ax, label='Contagem')
ax.set_xlabel('Confiança Prevista')
ax.set_ylabel('Acurácia Observada')
ax.set_title(f'Reliability Diagram (pós-Platt)\nECE={ece_cal:.4f}', fontweight='bold')
ax.legend(fontsize=8); ax.set_xlim(0,1); ax.set_ylim(0,1)

# ── Confusion Matrix ──────────────────────────────────────────────────────
ax = axes[2]
cm = confusion_matrix(test_labels, test_preds, normalize='true')
sns.heatmap(cm, annot=True, fmt='.3f', cmap='Blues', ax=ax,
            xticklabels=['Fake','Real'], yticklabels=['Fake','Real'],
            annot_kws={'size': 13, 'weight': 'bold'}, linewidths=0.5)
ax.set_title(
    f'Matriz de Confusão (normalizada)\n'
    f'Macro F1={metrics["macro_f1"]:.4f} | MCC={metrics["mcc"]:.4f}',
    fontweight='bold'
)
ax.set_xlabel('Predito'); ax.set_ylabel('Real')

plt.suptitle(f'RoBERTa — {EXP_ID}', fontsize=13, fontweight='bold')
plt.tight_layout()
fig_path = FIG_DIR / f'{EXP_ID}_results.png'
plt.savefig(fig_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figura salva: {fig_path}')

## Salvar JSON de resultados

In [ ]:
# Remover bin_data do JSON principal (pesado, já está na figura)
metrics_to_save = {k: v for k, v in metrics.items() if k != 'bin_data_calibrated'}

json_path = OUTPUT_DIR / f'{EXP_ID}_results.json'
with open(json_path, 'w') as f:
    json.dump(metrics_to_save, f, indent=2, default=str)

print(f'JSON salvo: {json_path}')
print(f'\n{"="*60}')
print('RESUMO PARA COPIAR NO CADERNO DO TCC:')
print(f'Experimento  : {EXP_ID}')
print(f'Macro F1     : {metrics["macro_f1"]:.4f}')
print(f'MCC          : {metrics["mcc"]:.4f}')
print(f'ROC-AUC      : {metrics["roc_auc"]:.4f}')
print(f'ECE (bruto)  : {ece_raw:.4f}')
print(f'ECE (Platt)  : {ece_cal:.4f}')
print(f'Brier Score  : {metrics["brier"]:.4f}')
print(f'F1 Fake      : {metrics["f1_fake"]:.4f}')
print(f'F1 Real      : {metrics["f1_real"]:.4f}')
print(f'{"="*60}')
print('\n→ Baixe o JSON via Output > nexus_roberta/*.json')
print('→ Coloque em experiments/roberta/ no VSCode')
print('→ Execute: python scripts/consolidate_results.py')